# Week 5 — PTB-XL ECG forecasting

Predict the last three seconds of a twelve-lead recording from its first seven.
Read each explanation, run its code, then inspect the output before continuing.
The editable settings are below. The actual loading, training, evaluation, and
plotting use the project's shared source. Open locally from the repository, or
put the project on Colab/Drive (see the README).

The current configuration trains E1?E4 for 120 epochs on the full training and validation cohorts. Select a CUDA kernel or a Colab GPU. Set `run_training=False` to inspect data and saved results.

In [ ]:
from pathlib import Path

COLAB = 'google.colab' in __import__('sys').modules or 'COLAB_RELEASE_TAG' in __import__('os').environ
DRIVE_MOUNT = Path('/content/drive')
MY_DRIVE = DRIVE_MOUNT / 'MyDrive'
CONFIG = {
    # Folders and runtime
    'project_root': MY_DRIVE / 'HIS_Project' / 'ptbxl_assignment' if COLAB else None,
    'mount_drive': COLAB,
    'data_source': 'folder',          # folder | physionet
    'data_root': MY_DRIVE / 'Colab Notebooks' / 'PTB-XL' if COLAB else Path('PTB-XL'),
    'cache_dir': Path('/content/week5_cache') if COLAB else Path('data/week5'),
    'results_dir': MY_DRIVE / 'Colab Notebooks' / 'PTBXL_Week5_Results' if COLAB else Path('results'),
    'install_packages': COLAB,
    'device': 'auto',                 # auto | cpu | cuda
    'allow_full_cpu': False,
    'study_id': 'week5_full',               # use a new ID if the cohort or scientific settings change
    # Cohort, signal and optimization
    'mode': 'full',                  # pilot | medium | full
    'train_patients': 1500,
    'val_patients': None,
    'medium_train_patients': 1500,
    'medium_val_patients': None,      # all official fold 9 patients
    'sampling_rate': 100,
    'history_seconds': 7,
    'forecast_seconds': 3,
    'filter_low_hz': 0.5,
    'filter_high_hz': 40.0,
    'batch_size': 64,
    'epochs': 120,
    'smoke_test': False,
    'smoke_epochs': 2,
    'learning_rate': 0.001,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.2,
    'seed': 42,
    # Opt-in experiments
    'experiment_id': None,           # E1-E4, or None to use selected_model + filtering
    'selected_model': 'lstm',        # lstm | cnn_lstm
    'filtering': False,
    'run_training': True,
    'run_all_experiments': True,
    'resume': False,
    'use_static_features': False,    # optional F1 after all four F0 conditions
    'run_training_ablation': False,
    'smaller_train_patients': 225,
    'shorter_epochs': 1,
    'quality_records': 32,
    'run_occlusion': True,
    'occlusion_patients': 12,
    'occlusion_seconds': 1.0,
    'final_test': False,              # only after four completed, frozen, 120-epoch full runs
}

| Settings | What to change |
|---|---|
| `project_root`, `mount_drive`, `data_source`, `data_root`, `cache_dir`, `results_dir` | Locate the shared code and PTB-XL, choose existing files or the official download, and choose where caches/checkpoints go. For PhysioNet in Colab use a runtime path such as `/content/PTB-XL` for faster access. |
| `install_packages`, `device`, `allow_full_cpu`, `study_id` | Choose dependency setup and CPU/GPU, allow slow full CPU training only deliberately, and label results so changed recipes don't overwrite them. |
| `mode`, `train_patients`, `val_patients`, `medium_train_patients`, `medium_val_patients` | Change the development cohort by whole patients. Full ignores these limits. The official fold numbers never change. |
| `sampling_rate`, `history_seconds`, `forecast_seconds`, `filter_low_hz`, `filter_high_hz` | Signal geometry and bandpass edges. The primary comparison requires 100 Hz, 7 s history, 3 s future. Changing filter settings requires a new study ID. |
| `batch_size`, `epochs`, `smoke_test`, `smoke_epochs`, `learning_rate`, `hidden_size`, `num_layers`, `dropout`, `seed` | Records per batch, training budget/short override, optimizer step, model capacity, regularization and reproducible sampling/initialization. |
| `experiment_id`, `selected_model`, `filtering`, `run_training`, `run_all_experiments`, `resume` | Choose a primary condition explicitly or by model/filter; authorize training of one or all four, or resume an interrupted matching run. Completed matches are reused. |
| `use_static_features`, `run_training_ablation`, `smaller_train_patients`, `shorter_epochs` | Enable the optional age/sex match or the separate one-factor-at-a-time cohort and budget comparisons. |
| `quality_records`, `run_occlusion`, `occlusion_patients`, `occlusion_seconds`, `final_test` | Bound exploratory work and the number and width of validation perturbations; opt in to the final locked fold only after full validation. |

## 01 — Environment

PyTorch does tensor operations and training, NumPy/Pandas handle arrays and
metadata, WFDB reads ECG files, SciPy performs signal processing, and Matplotlib
draws the figures. CPU is sufficient to check the workflow; use a GPU for the
full study. **Change:** `device` or `project_root`.

In [ ]:
import os
import platform
import subprocess
import sys

from IPython.display import Image, Markdown, display

if COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

root = CONFIG['project_root']
if root is None:
    start = Path.cwd().resolve()
    root = next((p for p in (start, *start.parents) if (p / 'src/week5.py').is_file()), None)
if root is None or not (Path(root) / 'src/week5.py').is_file():
    raise FileNotFoundError("Set CONFIG['project_root'] to the directory containing src/week5.py.")
root = Path(root).expanduser().resolve()
if CONFIG['install_packages']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(root / 'requirements.txt')], check=True)
sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wfdb
from scipy import signal

from config import LEADS, device_for, read_json, study_dir, write_json
from data import ECGDataset, download, loader, metadata, prepare, read_wave, select_patients
from evaluate import baseline, final_evaluation, outputs, ranking_key, verify_evaluation, verify_lock
from leakage_audit import audit
from models import CNNLSTMForecaster, LSTMForecaster
from preprocessing import filter_history, inverse_scale, prepare_transforms
from report import markdown_table
from train import seed_everything, static_inputs
from visualize import matched_forecasts, occlusion_heatmap, preprocessing_figures, recording_figures
from week5 import (EXPERIMENTS, compare_validation, configure, epoch_budget, primary_comparison,
                   quality_sample, run_experiment, run_path, temporal_occlusion, training_ablation,
                   validate_final)

for key in ('data_root', 'cache_dir', 'results_dir'):
    value = Path(CONFIG[key]).expanduser()
    CONFIG[key] = value if value.is_absolute() else root / value
cfg = configure(CONFIG)
device = device_for(cfg)
print('Python:', platform.python_version(), '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
      '| Selected device:', device)
print('Project:', root, '| Mode:', cfg['run_mode'], '| Effective epochs:', epoch_budget(cfg))

The selected device actually drives the training loop. If the output says CPU,
keep the training budget small unless you deliberately want a long local run.

## 02 — Load PTB-XL v1.0.3

The [official release](https://physionet.org/content/ptb-xl/1.0.3/) provides ECG
and patient metadata plus annotation definitions. `filename_lr` is a relative
path in `records100/`: the WFDB header and signal share the base name and use
`.hea` and `.dat`. Choosing PhysioNet downloads metadata first and only the
selected training and validation waveform pairs. **Change:** `data_root` or
`data_source`. Test metadata counts below do not mean test signals were loaded.

In [ ]:
data_root = Path(cfg['data']['root'])
if CONFIG['data_source'] == 'physionet':
    download(data_root, records=[])
elif CONFIG['data_source'] != 'folder':
    raise ValueError("Choose folder or physionet for data_source.")
frame = metadata(data_root)
statements = pd.read_csv(data_root / 'scp_statements.csv')
patients = select_patients(frame, cfg)
selected = {s: frame.loc[frame.split.eq(s) & frame.patient_id.isin(patients[s])].copy()
            for s in ('train', 'val', 'test')}
if CONFIG['data_source'] == 'physionet':
    download(data_root, records=pd.concat([selected['train'], selected['val']]).filename_lr.tolist())
print(f'{len(frame):,} ECGs; {frame.patient_id.nunique():,} patients; '
      f'{len(frame.columns) - 1} source metadata columns; {len(statements)} SCP definitions.')
display(frame.groupby('strat_fold').agg(ECGs=('ecg_id', 'size'), patients=('patient_id', 'nunique')))
display(pd.DataFrame([
    {'split': key, 'selected patients': rows.patient_id.nunique(), 'selected ECGs': len(rows)}
    for key, rows in selected.items()
]).set_index('split'))
display(selected['train'][['ecg_id','patient_id','age','sex','strat_fold','filename_lr']].head(5))
example = next(selected['train'].itertuples(index=False))
wave = read_wave(example, data_root)   # calls wfdb.rdsamp and validates shape/rate/lead order/units
header = wfdb.rdheader(str(data_root / example.filename_lr))
print('Example ECG:', example.ecg_id, '| dimensions:', wave.shape, '| leads:', header.sig_name)

## 03 — Understand 1,000 samples in ten seconds

For sampling frequency $f_s$ and duration $T$, $N=f_sT=100\times10=1000$
samples per lead. $\Delta t=1/f_s=0.01$ s. There are 12,000 voltage values per
recording. Sample $i$ occurs at $i/f_s$: the indices are 0–999, so the last
timestamp is 9.99 seconds. The first second runs from 0.00 to 0.99 seconds.
**Inspect:** Lead II in both plots and the ten sample rows.

In [ ]:
hz, h, f = cfg['data']['sampling_rate'], cfg['data']['history'], cfg['data']['horizon']
out = study_dir(cfg) / cfg['run_mode']
out.mkdir(parents=True, exist_ok=True)

def show_figure(figure, filename):
    image_path = out / filename
    figure.savefig(image_path, dpi=135, bbox_inches='tight')
    plt.close(figure)
    display(Image(filename=str(image_path)))

print(f'Shape {wave.shape}, units {header.units[1]}, last time {(len(wave)-1)/hz:.2f} s')
display(pd.DataFrame({'sample_index': np.arange(10), 'time_s': np.arange(10)/hz,
                      'Lead_II_mV': wave[:10, 1]}))
overview, sampling = recording_figures(wave, hz, h)
show_figure(overview, 'observed_ecg.png')
show_figure(sampling, 'sampling_zoom.png')

The markers in the zoom are actual recorded sample times; the lines only
connect the points. The dashed line is the forecasting boundary.

## 04 — Focused exploratory analysis

Summarize missingness and demographics in the development metadata. We treat
ages outside 0–120 as invalid for F1; it uses training-median imputation and a
missingness indicator. Sex is reported by dataset code and uses the training
mode if missing/invalid. Height/weight are described but not model inputs here.
Amplitude percentiles describe outliers without deleting high-amplitude ECGs.
The exploratory waveform scan is bounded; preparation separately checks every
selected development ECG and rejects bad records with an audit, not silently.
**Change:** `quality_records`.

In [ ]:
dev = frame.loc[frame.split.ne('test')]
count = dev.isna().sum()
missing = pd.DataFrame({'missing': count, 'percent': 100 * count / len(dev)})
missing.to_csv(out / 'eda_missing.csv')
display(missing.loc[(missing.missing > 0) | missing.index.isin(['age','sex','height','weight'])]
        .sort_values('missing', ascending=False))
age = pd.to_numeric(dev.age, errors='coerce')
invalid_age = int((age.notna() & ~age.between(0, 120)).sum())
print('Development metadata: invalid ages:', invalid_age)
display(age.where(age.between(0, 120)).describe().to_frame('valid_age'))
display(dev.sex.value_counts(dropna=False).rename('records').to_frame())
display(dev.groupby('patient_id').size().describe().to_frame('ECGs per patient'))
valid, failures, amplitude = quality_sample(selected['train'], cfg, CONFIG['quality_records'])
print(f'Training sample quality: {valid} valid recordings; {len(failures)} failed checks.')
if len(failures):
    display(failures)
display(amplitude.round(3))
amplitude.to_csv(out / 'eda_amplitude_mV.csv')
frequency, psd = signal.welch(wave[:, 1], fs=hz, nperseg=256)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.semilogy(frequency, np.maximum(psd, 1e-15))
ax.set(title='Example Lead II: Welch power spectral density', xlabel='Hz', ylabel='mV²/Hz')
show_figure(fig, 'eda_lead_II_psd.png')

Welch averages spectra over segments. The plot describes where this ECG sample's
power lies; it does not establish an optimal filter cutoff or a clinical abnormality.

## 05 — Official patient-safe folds

Use folds 1–8 for training, 9 for validation, and 10 for final testing. Repeated
ECGs from one person are related; a split by recording could put that person
on both sides and inflate apparent generalization. In pilot/medium, sample
whole patients and keep all their records. Run these checks **before** fitting
scalers or imputers. **Change:** `mode` and the relevant patient limits.

In [ ]:
for id_col in ('patient_id', 'ecg_id'):
    groups = [set(d[id_col]) for d in selected.values()]
    assert all(not a & b for i, a in enumerate(groups) for b in groups[i+1:]), id_col
fold_check = frame.groupby('split').agg(patients=('patient_id','nunique'), ECGs=('ecg_id','size'))
display(fold_check.reindex(['train','val','test']))
print('No patient or ECG ID overlap. Fold 10 was counted from metadata only.')

## 06 — Forecasting windows

One recording produces one example. With $H=700$ and $F=300$,
$X=W[0:H,:]$ and $Y=W[H:H+F,:]$. Sample 700 belongs to the target. No
overlapping windows are constructed. Revisit the all-lead plot to see the
boundary. **Fixed for the primary comparison:** history_seconds and forecast_seconds.

In [ ]:
X = wave[:h]
Y = wave[h:h+f]
assert X.shape == (700, 12) and Y.shape == (300, 12)
print('Observed input:', X.shape, 'first time:', 0.0, 'last time:', (h-1)/hz)
print('Raw target:', Y.shape, 'first time:', h/hz, 'last time:', (h+f-1)/hz)
display(Image(filename=str(out / 'observed_ecg.png')))

## 07 — Compare the two preprocessing choices

P0 uses raw history. P1 uses a [Butterworth filter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html)
with prototype order 4 (eighth-order bandpass) in second-order sections,
forward only. Each lead's initial state uses its first observed value. The
filter may remove drift and high-frequency content but can shift timing or
distort QRS/ST morphology. Both inputs share a raw-training-history per-lead
scaler: $z_l=(x_l-\mu_l)/\sigma_l$, and $x_l=z_l\sigma_l+\mu_l$.
Fit $\mu_l,\sigma_l$ on training history only. The target is always the raw
future transformed with that same scaler. **Change:** filter_low_hz and
filter_high_hz, then rerun the matched pair under a new study ID.

In [ ]:
cache, manifest = prepare(cfg)
audit_result = audit(cfg, cache)
scaler = prepare_transforms(cache, cfg, 'P0')
assert scaler == prepare_transforms(cache, cfg, 'P1')
filtered = filter_history(X, cfg)
print('Saved leakage checks:', audit_result['checks'])
display(pd.DataFrame({'lead': LEADS, 'raw training mean mV': scaler['mean'],
                      'raw training SD mV': scaler['std']}))
overlay, detail = preprocessing_figures(X, filtered, hz)
show_figure(overlay, 'P0_P1_observed_overlays.png')
show_figure(detail, 'P0_P1_qrs_psd.png')

The overlay and spectrum show what filtering changes. Only the matched
validation runs below show whether the changes improve forecasts.

## 08 — PyTorch Dataset and DataLoader

A Dataset returns one normalized history and target. The DataLoader batches
these examples. Only training uses shuffled order; validation scores remain
aligned with their ECG and patient IDs. The primary setting F0 has no static
inputs. **Change:** `batch_size`.

In [ ]:
static, names, _ = static_inputs(cfg, cache, 'F0')
train_set = ECGDataset(cache, cfg, 'train', 'P0', static['train'])
val_set = ECGDataset(cache, cfg, 'val', 'P0', static['val'])
seed_everything(CONFIG['seed'], cfg['training']['threads'])
generator = torch.Generator().manual_seed(CONFIG['seed'])
train_loader = loader(train_set, cfg, training=True, generator=generator)
val_loader = loader(val_set, cfg)
sample, batch = train_set[0], next(iter(train_loader))
print('Sample x/y:', tuple(sample['x'].shape), tuple(sample['y'].shape),
      '| patient:', sample['patient_id'], '| ECG:', int(train_set.metadata.iloc[0].ecg_id))
batch_row = int(batch['index'][0])
print('Batch x/y:', tuple(batch['x'].shape), tuple(batch['y'].shape),
      '| batch size:', len(batch['x']), '| first patient:', int(batch['patient_id'][0]),
      '| first ECG:', int(train_set.metadata.iloc[batch_row].ecg_id))
assert torch.equal(val_set[0]['y'], ECGDataset(cache, cfg, 'val', 'P1', static['val'])[0]['y'])
demo_x = batch['x'][:2]

## 09 — LSTM forecaster

The [LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
reads a 12-wide vector $x_t$ at each time step. $h_t$ is the exposed hidden
state and $c_t$ is the cell state. For each gate $j$, $W_j$ weights inputs,
$U_j$ weights the previous hidden state, and $b_j$ is the combined bias.
$\sigma$ is sigmoid, $\tanh$ bounds values, and $\odot$ is elementwise multiplication.

$$
i_t=\sigma(W_ix_t+U_ih_{t-1}+b_i),\quad
f_t=\sigma(W_fx_t+U_fh_{t-1}+b_f),\quad
o_t=\sigma(W_ox_t+U_oh_{t-1}+b_o)
$$
$$
g_t=\tanh(W_gx_t+U_gh_{t-1}+b_g),\quad
c_t=f_t\odot c_{t-1}+i_t\odot g_t,\quad
h_t=o_t\odot\tanh(c_t).
$$

The input gate writes candidate memory $g_t$, the forget gate retains old
memory, and the output gate exposes it. The forecast head maps the last
top-layer hidden state to $300\times12$ values. Backpropagation through time
applies the chain rule through the observed sequence, summing gradients for
shared recurrent weights. **Change:** hidden_size, num_layers, dropout.

In [ ]:
lstm = LSTMForecaster(horizon=f, **cfg['model']).eval()
with torch.inference_mode():
    lstm_prediction = lstm(demo_x)
assert lstm_prediction.shape == (len(demo_x), 300, 12)
print(lstm)
print('Input:', tuple(demo_x.shape), '→ output:', tuple(lstm_prediction.shape))
print('Trainable parameters:', sum(p.numel() for p in lstm.parameters() if p.requires_grad))

## 10 — CNN-LSTM forecaster

1D convolution applies learned kernels to neighboring observed times:
$$u_{o,t}=b_o+\sum_i\sum_{k=0}^{K-1}w_{o,i,k}x_{i,t+k-p}.$$
Here $i,o$ are input/output channels, $K$ is kernel width, $p$ is padding and
$b_o$ is a bias. ReLU computes $\max(0,u)$. Padding preserves time before
average pooling halves it. The LSTM processes the shorter feature sequence;
the head still predicts $300\times12$ values. **Change:** hidden_size,
num_layers, dropout.

In [ ]:
cnn = CNNLSTMForecaster(horizon=f, **cfg['model']).eval()
with torch.inference_mode():
    trace = demo_x.transpose(1, 2)
    print('CNN input [batch, channels, time]:', tuple(trace.shape))
    for layer in cnn.convolutions:
        trace = layer(trace)
        print(type(layer).__name__, tuple(trace.shape))
    print('LSTM input [batch, time, channels]:', tuple(trace.transpose(1, 2).shape))
    cnn_prediction = cnn(demo_x)
assert cnn_prediction.shape == (len(demo_x), 300, 12)
print(cnn.lstm, cnn.head)
print('Output:', tuple(cnn_prediction.shape),
      '| trainable parameters:', sum(p.numel() for p in cnn.parameters() if p.requires_grad))

## 11 — Train with the shared loop

The forward pass gives normalized predictions. Mean squared error averages
squared normalized residuals. Backpropagation computes gradients; clipping
bounds their total norm at 1.0 before the AdamW update. Validation does not
update weights. The best epoch minimizes validation patient-macro MAE, with
RMSE as a tie-breaker. The best checkpoint is for scoring; the last checkpoint
also holds optimizer and RNG state for resuming. **Change:** `epochs`,
`smoke_test`, `smoke_epochs`, or `learning_rate`.

In [ ]:
chosen = CONFIG['experiment_id']
if chosen is None:
    pair = (CONFIG['selected_model'], 'P1' if CONFIG['filtering'] else 'P0')
    chosen = next((eid for eid, spec in EXPERIMENTS.items() if spec == pair), None)
if chosen not in EXPERIMENTS:
    raise ValueError('Choose E1–E4, or a valid selected_model and filtering.')
print('Illustrative untrained batch MSE:', float((lstm_prediction - batch['y'][:2]).square().mean()))
print('Selected:', chosen, '| budget:', epoch_budget(cfg), '| training enabled:', CONFIG['run_training'])
if CONFIG['run_training'] and not CONFIG['run_all_experiments']:
    run_experiment(cfg, cache, chosen, resume=CONFIG['resume'])
else:
    print('No single-condition training in this cell.')

## 12 — Four matched experiments

E1/E2 change filtering for LSTM; E3/E4 do the same for CNN-LSTM.
E1/E3 and E2/E4 compare architecture at a fixed filtering setting.
Every condition is ECG-only (F0), uses the same cohort and target, and shares
training settings and metrics. **Change:** `experiment_id` and `run_training`,
or deliberately enable `run_all_experiments`. A completed compatible run is reused.

In [ ]:
if CONFIG['run_training'] and CONFIG['run_all_experiments']:
    for eid in EXPERIMENTS:
        run_experiment(cfg, cache, eid, resume=CONFIG['resume'])
table, contrasts, measured = primary_comparison(cfg, cache)
display(table[['experiment','architecture','filter','status','best_epoch']])
histories = []
for eid in measured:
    log = pd.read_csv(run_path(cfg, eid) / 'history.csv').assign(experiment=eid)
    histories.append(log)
history_table = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
if histories:
    display(history_table[['experiment','epoch','train_mse_normalized','val_mse_normalized',
                           'val_mae_mV','val_rmse_mV','seconds']].tail(12))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for eid, log in history_table.groupby('experiment'):
        axes[0].plot(log.epoch, log.train_mse_normalized, '.-', label=eid)
        axes[1].plot(log.epoch, log.val_patient_macro_mae_mV, '.-', label=eid)
    axes[0].set(xlabel='Epoch', ylabel='Training MSE (normalized)')
    axes[1].set(xlabel='Epoch', ylabel='Validation patient MAE (mV)')
    axes[1].legend()
    show_figure(fig, 'training_curves.png')
else:
    print('No compatible training history yet.')

## 13 — Evaluate in original millivolts

After inverse scaling, let $e=\hat y-y$. MAE is $\mathrm{mean}(|e|)$ in mV,
MSE is $\mathrm{mean}(e^2)$ in mV² and RMSE is $\sqrt{\mathrm{MSE}}$ in mV.
RMSE emphasizes larger errors. Patient-macro MAE averages record MAEs within
each patient before equally weighting patients. Per-lead metrics reduce the
record/time axes. Persistence repeats the last observed sample.
Repeat-window copies the last 300 observed samples.

In [ ]:
baseline_rows = []
for name in ('persistence', 'repeat_window'):
    folder = outputs(cfg, name)
    if (folder / 'val_receipt.json').exists():
        verify_evaluation(folder, 'val')
        row = read_json(folder / 'val.json')
        if row['cohort'] != manifest['cohort']:
            raise ValueError('Saved baseline belongs to another cohort.')
    else:
        row = baseline(cfg, cache, name)
    baseline_rows.append(row)
baselines = pd.DataFrame(baseline_rows)[['name','mae_mV','rmse_mV','patient_macro_mae_mV']]
display(baselines)
if measured:
    exemplar = chosen if chosen in measured else next(iter(measured))
    folder = outputs(cfg, measured[exemplar]['name'])
    print('Per-lead validation errors for', exemplar)
    display(pd.read_csv(folder / 'val_leads.csv')[['lead','mae_mV','rmse_mV']])
else:
    print('Neural per-lead metrics are pending.')

## 14 — Answer the comparisons with measured evidence

Improvement is $100(E_{\mathrm{reference}}-E_{\mathrm{candidate}})/E_{\mathrm{reference}}$:
positive is better; negative is worse. Here $E$ is patient-macro MAE. The paired
bootstrap resamples validation patients and estimates candidate-minus-reference
error in mV. An interval crossing zero is inconclusive. Parameter counts and
runtime are reported with the architecture results.

The optional ablation changes either cohort size or budget and uses identical
validation ECGs. To score the final test, finish all four full-data F0
120-epoch runs and enable final_test. The notebook freezes the four validation
checkpoints before opening test waveforms.

In [ ]:
display(table)
display(contrasts.drop(columns=['reference', 'candidate'], errors='ignore'))
winner = min(measured, key=lambda eid: ranking_key(measured[eid])) if len(measured) == 4 else None
if winner:
    print(f'Lowest measured validation patient-macro MAE: {winner} '
          f'({measured[winner]["patient_macro_mae_mV"]:.6f} mV).')
else:
    print('Overall ranking pending until all four conditions are measured.')

ablation_file = out / 'week5_ablation.csv'
ablation = pd.DataFrame()
if winner and CONFIG['run_training'] and CONFIG['run_training_ablation']:
    ablation = training_ablation(cfg, cache, winner, CONFIG['smaller_train_patients'],
                                 CONFIG['shorter_epochs'], resume=CONFIG['resume'])
    ablation.to_csv(ablation_file, index=False)
if not ablation.empty:
    display(ablation)
else:
    print('Cohort/epoch ablation not run in this execution.')

test_summary = pd.DataFrame()
if CONFIG['final_test']:
    validate_final(cfg, cache)
    verify_lock(cfg, cache)
    if CONFIG['data_source'] == 'physionet':
        download(data_root, records=selected['test'].filename_lr.tolist())
    ledger = final_evaluation(cfg, cache)
    test_summary = pd.DataFrame([read_json(outputs(cfg, name) / 'test.json')
                                 for name in ledger['conditions']])
    display(test_summary[['name','mae_mV','rmse_mV','patient_macro_mae_mV']])
else:
    print('Final test remains locked.')

Best measured validation error is a ranking on this cohort. A small difference
or an interval crossing zero is insufficient for a broad superiority claim.

## 15 — Compare actual and predicted twelve-lead ECGs

For each architecture take its better validation filter condition, then plot
both models on the **lowest common validation ECG ID**. This rule is fixed
before judging the waveforms. Lead II's first detected future QRS is enlarged
if it fits; detection is used only for display. The model did not see that future.

In [ ]:
example_id = None
if winner:
    selected_models = [min(ids, key=lambda eid: ranking_key(measured[eid]))
                       for ids in (('E1','E2'), ('E3','E4'))]
    example_id, forecast, inset = matched_forecasts(
        cfg, cache, {eid: measured[eid]['name'] for eid in selected_models}
    )
    print('ECG:', example_id, '| selected models:', selected_models)
    show_figure(forecast, 'week5_forecasts.png')
    if inset is not None:
        show_figure(inset, 'week5_qrs.png')
    else:
        print('No future QRS met the predefined zoom rule for this ECG.')
else:
    print('Forecast comparison awaits all four validation conditions.')

The common recording reveals whether the predictions capture amplitude and
timing on one example. It cannot establish morphology fidelity for the dataset.

## 16 — Optional age and sex comparison

Take the best F0 condition only after all four are measured. F1 adds age, sex
and their missingness indicators, fitted on training only. A learned embedding
is repeated along the observed time axis and concatenated before the first
temporal layer. Keep the architecture, filter, cohort and optimization budget
fixed. **Change:** `use_static_features` and `run_training`.
Week 4 mRMR/permutation extensions remain outside this notebook.

In [ ]:
age_sex = pd.DataFrame()
if winner and CONFIG['use_static_features']:
    from week5 import validation_result
    f1_score = validation_result(cfg, cache, winner, 'F1')
    if not f1_score and CONFIG['run_training']:
        _, f1_score = run_experiment(cfg, cache, winner, features='F1', resume=CONFIG['resume'])
    if f1_score:
        age_sex = pd.DataFrame([compare_validation(cfg, measured[winner], cfg, f1_score)])
        age_sex.to_csv(out / 'week5_age_sex.csv', index=False)
        print('F0 pooled MAE/RMSE:', measured[winner]['mae_mV'], measured[winner]['rmse_mV'])
        display(age_sex)
if age_sex.empty:
    print('Matched age/sex comparison is pending or disabled.')

## 17 — Explainability: temporal occlusion sensitivity

Use the validation-selected F0 checkpoint and one ECG per patient, taking the
first distinct patients in saved ECG-ID order. Mask one interval in one lead by
setting its normalized input to zero, equivalent to the raw training mean.
Keep the target and all other inputs fixed. Sensitivity is
$I(S)=E(X^{(S)})-E(X)$ in mV. Red means error rose; blue means it fell. Artificial
mask boundaries can affect the result. This is sensitivity, not clinical
causality. **Change:** occlusion_patients and occlusion_seconds.

In [ ]:
importance, occlusion = pd.DataFrame(), None
if winner and CONFIG['run_occlusion']:
    importance, occlusion = temporal_occlusion(
        cfg, cache, run_path(cfg, winner), patients=CONFIG['occlusion_patients'],
        window_seconds=CONFIG['occlusion_seconds'],
    )
    importance.to_csv(out / 'week5_occlusion.csv')
    write_json(out / 'week5_occlusion.json', occlusion)
    show_figure(occlusion_heatmap(importance), 'week5_occlusion.png')
    hottest = importance.stack().idxmax()
    print(f'{occlusion["patients"]} paired validation patients; largest increase: '
          f'{hottest[0]} {hottest[1]}, {importance.loc[hottest[0], hottest[1]]:.6f} mV.')
else:
    print('Temporal occlusion awaits complete validation selection or is disabled.')

## 18 — Final Week 5 conclusions

Generate a short report from this execution and the verified validation
artifacts. Pending cells stay pending. Report final test metrics separately;
one seed, one validation cohort, and a short run do not support a clinical or
population-wide forecast claim.

In [ ]:
table.to_csv(out / 'week5_validation.csv', index=False)
contrasts.to_csv(out / 'week5_contrasts.csv', index=False)
baselines.to_csv(out / 'week5_baselines.csv', index=False)
if not test_summary.empty:
    test_summary.to_csv(out / 'week5_final_test.csv', index=False)

def verdict(index):
    row = contrasts.iloc[index]
    if row['interpretation'] == 'pending':
        return f'{row["comparison"]}: pending.'
    return (f'{row["comparison"]}: {row["improvement_pct"]:+.2f}% reduction in patient MAE; '
            f'paired difference {row["difference_mV"]:+.6f} mV '
            f'(95% interval {row["CI_low_mV"]:+.6f} to {row["CI_high_mV"]:+.6f}), '
            f'{row["interpretation"]}.')

training_text = (f'{len(measured)} of four conditions completed {epoch_budget(cfg)} epochs. '
                 'Best epochs and per-epoch losses are reported above.')
if not ablation.empty:
    training_text += '\n\n' + markdown_table(ablation[['change','improvement_pct','interpretation']])
else:
    training_text += ' Cohort-size and epoch-budget effects are pending in this execution.'
occlusion_text = (f'{hottest[0]} during {hottest[1]} produced the highest measured error change '
                  f'({importance.loc[hottest[0], hottest[1]]:+.6f} mV) '
                  f'across {occlusion["patients"]} selected validation patients.'
                  if globals().get('occlusion') else 'Temporal occlusion results are pending.')
sections = [
    ('Dataset and number of patients', f'{len(frame):,} ECGs, {frame.patient_id.nunique():,} patients.'),
    ('Patient-safe fold validation', 'No patient or ECG IDs overlap.\n\n' + markdown_table(
        fold_check.reset_index().rename(columns={'index': 'split'}))),
    ('Missing data and preprocessing', f'{invalid_age} invalid ages in development metadata. '
     'P0 and P1 share a raw-training-history scaler and identical unfiltered targets.'),
    ('LSTM architecture', f'{cfg["model"]["depth"]} layer(s), {cfg["model"]["hidden"]} hidden units, '
     f'{sum(p.numel() for p in lstm.parameters() if p.requires_grad):,} trainable F0 parameters.'),
    ('CNN-LSTM architecture', 'Two convolution blocks, average pooling, LSTM and shared forecast geometry. '
     f'{sum(p.numel() for p in cnn.parameters() if p.requires_grad):,} trainable F0 parameters.'),
    ('Training and convergence', training_text),
    ('MAE/RMSE calculation', 'Forecast and target are compared in mV (MSE in mV²); patient-macro MAE '
     'weights patients equally.\n\n' + markdown_table(table)),
    ('LSTM preprocessing comparison', verdict(0)),
    ('CNN-LSTM preprocessing comparison', verdict(1)),
    ('LSTM versus CNN-LSTM', verdict(2) + ' ' + verdict(3)),
    ('Twelve-lead actual versus predicted ECG', f'Same validation ECG: {example_id}.' if example_id else 'Pending.'),
    ('Optional age/sex experiment', markdown_table(age_sex)),
    ('Temporal occlusion explainability', occlusion_text + ' Perturbation sensitivity does not imply causality.'),
    ('Limitations and conclusions', (f'{winner} has the lowest measured validation patient-macro MAE. '
     if winner else 'The lowest-error configuration is pending. ') +
     ('Final test results are separate.' if not test_summary.empty else 'Final test remains locked. ') +
     'One seed and this training budget do not establish broad superiority.'),
]
report = '# Week 5 research conclusions\n\n' + '\n\n'.join(
    f'## {index}. {title}\n\n{body}' for index, (title, body) in enumerate(sections, 1)
)
(out / 'WEEK5_RESULTS.md').write_text(report + '\n', encoding='utf-8')
display(Markdown(report))
print('Results saved under:', out)